# 02 — Data Transformation
**Wildfire Risk Modeling Exercise | Cell2Fire W — Scott & Burgan**

Transforms raw exercise data into a C2F-W instance folder.
Every assumption is documented here for the report's **Workflow & Assumptions** section.

| Step | Input | Output | Assumption |
|---|---|---|---|
| 1 | `Forest_elevation.tif` (EPSG:4326) | Reference grid (EPSG:5070, ~27m) | Bilinear resampling |
| 2 | `Forest_SB40.tif` | `fuels.asc` | Nearest-neighbour (categorical) |
| 3 | `Forest_elevation.tif` | `elevation.asc` | Bilinear resampling |
| 4 | `Forest_depth.tif`, `Forest_rhof1.tif`, `Forest_moist1.tif` | CBH, CBD, FMC layers | Bilinear; FMC fraction→% |
| 5 | Aligned elevation | Slope & aspect | Sobel filter; boundary eroded 1px |
| 6 | CBD | CCF | 95th-percentile normalisation |
| 7 | `Forest_Synoptic_Weather_Data.csv` | `Weather.csv` | 10-min→hourly mean; mph→km/h |
| 8 | `Forest_ignition.geojson` | `Ignitions.csv` | WGS84→EPSG:5070→row-major cell index |
| 9 | C2F-W `--gen-data` run | `Data.csv` patched | Only burnable cells updated |

In [ ]:
# ── Town selection — change to run Prairie
TOWN = "forest"   # "forest" or "prairie"

In [ ]:
# ============================================================
# CELL 1 — Imports, config, paths
# ============================================================
import sys, pathlib, shutil, tempfile, subprocess, json
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from pyproj import Transformer
from scipy.ndimage import sobel, binary_erosion

REPO_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from utils import load_config

cfg      = load_config(TOWN, REPO_ROOT)
RAW      = REPO_ROOT / cfg["data_raw"]
INSTANCE = REPO_ROOT / cfg["instance_dir"]
RESULTS  = REPO_ROOT / cfg["results_dir"]
INSTANCE.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

f        = cfg["files"]
sim_cfg  = cfg["simulation"]
wx_cfg   = cfg["weather"]
CRS_OUT  = cfg["crs_out"]

SURF     = RAW / f"{TOWN}-surface-fuels-and-surface-data"
IGN_DIR  = RAW / f"{TOWN}-ignition"
WX_DIR   = RAW / f"{TOWN}-weather-data"

C2FW_DIR = REPO_ROOT / "C2F-W"
BINARY   = C2FW_DIR / "Cell2Fire" / "Cell2Fire"
LOOKUP   = C2FW_DIR / "data" / "ScottAndBurgan" / "Zona_60-tif" / "spain_lookup_table.csv"

print(f"Town:     {cfg['display_name']}")
print(f"Instance: {INSTANCE}")
print(f"Binary:   {'✓' if BINARY.exists() else '✗ MISSING'} {BINARY}")
print(f"Lookup:   {'✓' if LOOKUP.exists() else '✗ MISSING'} {LOOKUP.name}")

In [ ]:
# ============================================================
# CELL 2 — Helper functions
# ============================================================

def write_asc(tif_path, asc_path, integer=False):
    """Convert a single-band GeoTIFF to ESRI ASCII raster."""
    with rasterio.open(tif_path) as ds:
        data = ds.read(1)
        nodata = ds.nodata
        t = ds.transform
        nrows, ncols = data.shape
        xll = t.c
        yll = t.f + t.e * nrows
        cellsize = t.a
    with open(asc_path, "w") as fh:
        fh.write(f"ncols         {ncols}\n")
        fh.write(f"nrows         {nrows}\n")
        fh.write(f"xllcorner     {xll:.6f}\n")
        fh.write(f"yllcorner     {yll:.6f}\n")
        fh.write(f"cellsize      {cellsize:.6f}\n")
        fh.write(f"NODATA_value  {int(nodata) if integer else nodata}\n")
        np.savetxt(fh, data, fmt="%d" if integer else "%.4f")
    return nrows, ncols, cellsize

def read_asc(path):
    """Read ESRI ASCII raster, return (2D array, header dict)."""
    header = {}
    with open(path) as fh:
        for _ in range(6):
            key, val = fh.readline().split()
            header[key.lower()] = float(val)
        data = np.loadtxt(fh)
    return data, header

def reproject_to_ref(src_path, ref_meta, resampling=Resampling.bilinear):
    """Reproject a raster to match ref_meta, return 2D float32 array."""
    dst = np.full((ref_meta["height"], ref_meta["width"]), np.nan, dtype=np.float32)
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref_meta["transform"], dst_crs=ref_meta["crs"],
            resampling=resampling,
            src_nodata=src.nodata, dst_nodata=np.nan,
        )
    return dst

def save_tif(array, path, ref_meta, dtype="float32", nodata=-9999):
    """Save 2D array as GeoTIFF aligned to reference grid."""
    meta = ref_meta.copy()
    meta.update(dtype=dtype, nodata=nodata, count=1)
    arr = array.copy().astype(np.float32)
    arr[np.isnan(arr)] = nodata
    with rasterio.open(path, "w", **meta) as dst:
        dst.write(arr, 1)

print("✓ Cell 2 ready — helpers defined")

In [ ]:
# ============================================================
# CELL 3 — Build reference grid, write elevation.asc + fuels.asc
# Assumption: elevation drives the reference grid (bilinear).
# SB40 resampled with nearest-neighbour (categorical data).
# 32767 and negative SB40 codes → 0 (non-burnable).
# ============================================================

# ── Reference grid from elevation ────────────────────────────
with rasterio.open(SURF / f["elevation"]) as src:
    transform, width, height = calculate_default_transform(
        src.crs, CRS_OUT, src.width, src.height, *src.bounds
    )
ref_meta = {
    "driver": "GTiff", "crs": CRS_OUT,
    "transform": transform, "width": width, "height": height,
    "count": 1, "dtype": "float32", "nodata": -9999,
}
print(f"Reference grid: {height}\u00d7{width} px | "
      f"res={transform.a:.1f}m | CRS={CRS_OUT}")
print("─" * 55)

# ── Elevation → elevation.tif + elevation.asc ─────────────────
print("\n[1/2] Elevation")
elev_tif = INSTANCE / "elevation.tif"
with rasterio.open(SURF / f["elevation"]) as src:
    with rasterio.open(elev_tif, "w", **ref_meta) as dst:
        reproject(
            source=rasterio.band(src, 1),
            destination=rasterio.band(dst, 1),
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=transform, dst_crs=CRS_OUT,
            resampling=Resampling.bilinear,
        )
write_asc(elev_tif, INSTANCE / "elevation.asc", integer=False)
with rasterio.open(elev_tif) as ds:
    e = ds.read(1)
    e = e[e != -9999]
print(f"  ✓ elevation.asc | min={e.min():.1f}m  max={e.max():.1f}m")

# ── SB40 → fuels.tif + fuels.asc ─────────────────────────────
print("\n[2/2] SB40 Fuel Model")
with rasterio.open(SURF / f["sb40"]) as src:
    data = src.read(1)
    data[data == 32767] = 0   # urban/water sentinel → non-burnable
    data[data < 0]      = 0
    tmp = pathlib.Path(tempfile.mktemp(suffix=".tif"))
    tmp_meta = src.meta.copy()
    tmp_meta.update(nodata=0)
    with rasterio.open(tmp, "w", **tmp_meta) as t:
        t.write(data, 1)

fuel_tif = INSTANCE / "fuels.tif"
fuel_meta = ref_meta.copy()
fuel_meta.update(dtype="int16", nodata=0)
with rasterio.open(tmp) as src:
    with rasterio.open(fuel_tif, "w", **fuel_meta) as dst:
        reproject(
            source=rasterio.band(src, 1),
            destination=rasterio.band(dst, 1),
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=transform, dst_crs=CRS_OUT,
            resampling=Resampling.nearest,
        )
tmp.unlink()
write_asc(fuel_tif, INSTANCE / "fuels.asc", integer=True)

fuels_data, fuels_hdr = read_asc(INSTANCE / "fuels.asc")
burnable_pct = (fuels_data > 0).sum() / fuels_data.size * 100
print(f"  ✓ fuels.asc | burnable={burnable_pct:.1f}%  "
      f"non-burnable={100-burnable_pct:.1f}%")

print("\n✓ Cell 3 ready — elevation.asc and fuels.asc written")

In [ ]:
# ============================================================
# CELL 4 — Reproject canopy layers to reference grid
# CBD  : Rhof 1hr (kg/m³) — canopy bulk density
# CBH  : Depth (m)        — canopy base height
# FMC  : Moist 1hr (frac) — fuel moisture, fraction → integer %
# CCF  : derived from CBD — canopy cover fraction (0-100)
# Assumption: all use bilinear resampling; FMC clipped 1-200%.
# CCF normalised to 95th percentile of non-zero CBD values.
# ============================================================

print("Reprojecting canopy layers...")
print("─" * 55)

# CBD
print("[1/4] Canopy Bulk Density (Rhof 1hr → CBD)")
cbd_arr = reproject_to_ref(SURF / f["rhof1"], ref_meta)
cbd_arr = np.clip(cbd_arr, 0, None)
save_tif(cbd_arr, INSTANCE / "cbd.tif", ref_meta)
print(f"  ✓ min={np.nanmin(cbd_arr):.3f}  max={np.nanmax(cbd_arr):.3f} kg/m\u00b3")

# CBH
print("[2/4] Canopy Base Height (Depth → CBH)")
cbh_arr = reproject_to_ref(SURF / f["depth"], ref_meta)
cbh_arr = np.clip(cbh_arr, 0, None)
save_tif(cbh_arr, INSTANCE / "cbh.tif", ref_meta)
print(f"  ✓ min={np.nanmin(cbh_arr):.3f}  max={np.nanmax(cbh_arr):.3f} m")

# FMC
print("[3/4] Fuel Moisture Content (Moist 1hr → FMC %)")
fmc_arr = reproject_to_ref(SURF / f["moist1"], ref_meta)
fmc_pct = np.clip((fmc_arr * 100).round(), 1, 200)
fmc_pct[np.isnan(fmc_arr)] = np.nan
save_tif(fmc_pct, INSTANCE / "fmc.tif", ref_meta)
print(f"  ✓ min={np.nanmin(fmc_pct):.0f}%  max={np.nanmax(fmc_pct):.0f}%")

# CCF
print("[4/4] Canopy Cover Fraction (derived from CBD)")
cbd_pos = cbd_arr[cbd_arr > 0]
cbd_max = np.nanpercentile(cbd_pos, 95) if len(cbd_pos) else 1.0
ccf_arr = np.clip((cbd_arr / cbd_max) * 100, 0, 100).round()
ccf_arr[np.isnan(cbd_arr)] = np.nan
save_tif(ccf_arr, INSTANCE / "ccf.tif", ref_meta)
print(f"  ✓ min={np.nanmin(ccf_arr):.0f}%  max={np.nanmax(ccf_arr):.0f}%")

print("\n✓ Cell 4 ready — canopy layers written")

In [ ]:
# ============================================================
# CELL 5 — Derive slope & aspect from aligned elevation
# Assumption: Sobel filter on EPSG:5070 elevation (metres).
# Boundary erosion (1px) removes Sobel edge artifacts.
# Both saved as TIFs for Data.csv patching.
# ============================================================

with rasterio.open(INSTANCE / "elevation.tif") as ds:
    elev_aligned = ds.read(1).astype(float)
    elev_aligned[elev_aligned == -9999] = np.nan
    cs = ds.transform.a   # cellsize in metres

valid        = ~np.isnan(elev_aligned)
elev_filled  = elev_aligned.copy()
elev_filled[~valid] = 0

dz_dx      = sobel(elev_filled, axis=1) / (8 * cs)
dz_dy      = sobel(elev_filled, axis=0) / (8 * cs)
slope_pct  = np.sqrt(dz_dx**2 + dz_dy**2) * 100
aspect_deg = (np.degrees(np.arctan2(-dz_dy, dz_dx)) + 360) % 360

# Erode valid mask by 1px to remove Sobel boundary artifacts
valid_eroded           = binary_erosion(valid, iterations=1)
slope_pct[~valid_eroded]  = np.nan
aspect_deg[~valid_eroded] = np.nan

save_tif(slope_pct,  INSTANCE / "slope.tif",  ref_meta)
save_tif(aspect_deg, INSTANCE / "aspect.tif", ref_meta)

print(f"Slope:  min={np.nanmin(slope_pct):.1f}%  "
      f"max={np.nanmax(slope_pct):.1f}%  "
      f"mean={np.nanmean(slope_pct):.1f}%")
print(f"Aspect: min={np.nanmin(aspect_deg):.1f}\u00b0  "
      f"max={np.nanmax(aspect_deg):.1f}\u00b0")
print("\n✓ Cell 5 ready — slope.tif and aspect.tif written")

In [ ]:
# ============================================================
# CELL 6 — Weather.csv and Ignitions.csv
# Weather: 10-min obs → hourly mean; mph → km/h.
# Ignitions: WGS84 lon/lat → EPSG:5070 → row-major cell index.
# ============================================================

station_id   = wx_cfg["station_id"]
resample_freq = wx_cfg["resample_freq"]

# ── Weather ──────────────────────────────────────────────────
df_raw = pd.read_csv(WX_DIR / f["weather"], comment="#")
if station_id:
    df_raw = df_raw[
        df_raw.iloc[:, 0].astype(str).str.startswith(str(station_id), na=False)
    ].copy()
df_raw.columns = ["Station_ID", "Date_Time", "temp_F", "RH",
                  "ws_mph", "WD", "gust_mph"]
for col in ["temp_F", "RH", "ws_mph", "WD", "gust_mph"]:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")
df_raw["Date_Time"] = pd.to_datetime(df_raw["Date_Time"]).dt.tz_localize(None)
df_raw = df_raw.set_index("Date_Time").sort_index()

df_hr = df_raw[["ws_mph", "WD"]].resample(resample_freq).mean()
df_hr["WS"]          = (df_hr["ws_mph"] * 1.60934).round(1)  # mph → km/h
df_hr["WD"]          = df_hr["WD"].round(1)
df_hr["Scenario"]    = f"{TOWN}_exercise"
df_hr["datetime"]    = df_hr.index.strftime("%Y-%m-%d")
df_hr["FireScenario"]= 1

wx_out = df_hr[["Scenario", "datetime", "WS", "WD",
                "FireScenario"]].reset_index(drop=True)
wx_out.to_csv(INSTANCE / "Weather.csv", index=False)

print("── Weather.csv ──────────────────────────────────────")
print(f"  {len(df_raw)} rows ({df_raw.index[1]-df_raw.index[0]}) "
      f"\u2192 {len(wx_out)} hourly rows")
print(f"  WS: {wx_out['WS'].min()}\u2013{wx_out['WS'].max()} km/h  "
      f"WD: {wx_out['WD'].min()}\u2013{wx_out['WD'].max()}\u00b0")
print(f"  \u2713 Weather.csv written")
print(wx_out.to_string(index=False))

# ── Ignitions ─────────────────────────────────────────────────
with open(IGN_DIR / f["ignition"]) as fh:
    ign = json.load(fh)
lon, lat = ign["features"][0]["geometry"]["coordinates"]

tr = Transformer.from_crs("EPSG:4326", CRS_OUT, always_xy=True)
x_proj, y_proj = tr.transform(lon, lat)

ncols    = int(fuels_hdr["ncols"])
nrows    = int(fuels_hdr["nrows"])
xll      = fuels_hdr["xllcorner"]
yll      = fuels_hdr["yllcorner"]
cellsize = fuels_hdr["cellsize"]

col_idx     = int((x_proj - xll) / cellsize)
row_idx     = int(nrows - (y_proj - yll) / cellsize)
cell_number = row_idx * ncols + col_idx + 1
fuel_at_ign = int(fuels_data[row_idx, col_idx])

# Store for notebook 04
IGN_X, IGN_Y = x_proj, y_proj
left   = xll;           right  = xll + cellsize * ncols
bottom = yll;           top    = yll + cellsize * nrows
extent = [left, right, bottom, top]

pd.DataFrame({"Year": [1], "Ncell": [cell_number]}).to_csv(
    INSTANCE / "Ignitions.csv", index=False)

print(f"\n── Ignitions.csv ────────────────────────────────────")
print(f"  WGS84  : {lat:.4f}\u00b0N, {abs(lon):.4f}\u00b0W")
print(f"  {CRS_OUT}: x={x_proj:.1f}, y={y_proj:.1f}")
print(f"  Grid   : row={row_idx}, col={col_idx} \u2192 cell {cell_number} of {nrows*ncols}")
print(f"  Fuel at ignition: {fuel_at_ign} "
      f"{'\u2713 burnable' if fuel_at_ign > 0 else '\u2717 NON-BURNABLE!'}")
print(f"  \u2713 Ignitions.csv written")

print("\n\u2713 Cell 6 ready \u2014 Weather.csv and Ignitions.csv written")

In [ ]:
# ============================================================
# CELL 7 — Copy lookup table; list instance contents
# Note: TIF files stay for Data.csv patching in Cell 8.
# They will be removed AFTER patching.
# ============================================================

shutil.copy(LOOKUP, INSTANCE / "spain_lookup_table.csv")
print(f"\u2713 spain_lookup_table.csv copied")

print(f"\n{'File':<35} {'Size':>9}")
print("\u2500" * 46)
for fp in sorted(INSTANCE.iterdir()):
    print(f"  {fp.name:<33} {fp.stat().st_size/1024:>7.1f} KB")

print("\n\u2713 Cell 7 ready \u2014 lookup table copied, instance contents listed")

In [ ]:
# ============================================================
# CELL 8 — Generate base Data.csv via --gen-data, then patch
# C2F-W writes Data.csv on first run with --gen-data.
# We then enrich it with: slope(ps), aspect(saz),
# cbd, cbh, fmc, ccf — only for burnable cells.
# ============================================================

# ── Step 1: Run --gen-data ────────────────────────────────────
print("[1/3] Generating base Data.csv via --gen-data...")
shutil.rmtree(RESULTS, ignore_errors=True)
RESULTS.mkdir()

r = subprocess.run([
    str(BINARY),
    "--input-instance-folder", str(INSTANCE),
    "--output-folder",         str(RESULTS),
    "--sim",        sim_cfg["sim_model"],
    "--nsims",      "1",
    "--nthreads",   str(sim_cfg["nthreads"]),
    "--weather",    sim_cfg["weather_mode"],
    "--ignitions",
    "--gen-data",
    "--Fire-Period-Length", "1.0",
    "--ROS-CV",     "0.0",
    "--seed",       str(sim_cfg["seed"]),
], capture_output=True, text=True)

if r.returncode != 0:
    print("ERROR:", r.stderr[-800:])
    raise RuntimeError("--gen-data failed")
print(f"  \u2713 Data.csv generated")

# ── Step 2: Load Data.csv and TIF arrays ─────────────────────
print("\n[2/3] Loading Data.csv and reprojected layers...")
df_data = pd.read_csv(INSTANCE / "Data.csv")
print(f"  Rows: {len(df_data):,}  Cols: {list(df_data.columns)}")

def load_tif_flat(path):
    with rasterio.open(path) as ds:
        arr = ds.read(1).astype(float)
        arr[arr == ds.nodata] = np.nan
    return arr.flatten()

fuels_flat  = fuels_data.flatten()
slope_flat  = load_tif_flat(INSTANCE / "slope.tif")
aspect_flat = load_tif_flat(INSTANCE / "aspect.tif")
cbd_flat    = load_tif_flat(INSTANCE / "cbd.tif")
cbh_flat    = load_tif_flat(INSTANCE / "cbh.tif")
fmc_flat    = load_tif_flat(INSTANCE / "fmc.tif")
ccf_flat    = load_tif_flat(INSTANCE / "ccf.tif")

# ── Step 3: Patch enriched columns (burnable cells only) ──────
print("\n[3/3] Patching Data.csv columns...")

def patch(df, col, values):
    df[col] = np.where(fuels_flat > 0, values, np.nan)
    return df

df_data = patch(df_data, "ps",  slope_flat)
df_data = patch(df_data, "saz", aspect_flat)
df_data = patch(df_data, "cbd", cbd_flat)
df_data = patch(df_data, "cbh", cbh_flat)
df_data = patch(df_data, "fmc", fmc_flat)
df_data = patch(df_data, "ccf", ccf_flat)
df_data.to_csv(INSTANCE / "Data.csv", index=False)

# ── Remove intermediate TIFs now that patching is done ────────
for tif in INSTANCE.glob("*.tif"):
    tif.unlink()
print("  \u2713 Intermediate TIFs removed")

# ── Verify ────────────────────────────────────────────────────
burnable = df_data[df_data["fueltype"] != "NF"]
print(f"\n\u2500" * 50)
print(f"  Burnable cells: {len(burnable):,} of {len(df_data):,}")
for col, label in [("ps","slope %"), ("saz","aspect \u00b0"),
                   ("cbd","kg/m\u00b3"), ("cbh","m"),
                   ("fmc","fmc %"), ("ccf","ccf %")]:
    filled = burnable[col].notna().sum()
    pct = filled / len(burnable) * 100 if len(burnable) else 0
    print(f"  {'\u2713' if pct > 80 else '\u26a0'}  "
          f"{col:<4} ({label:<10}): "
          f"{burnable[col].min():.2f} \u2013 {burnable[col].max():.2f} "
          f"({pct:.0f}% filled)")

print("\n\u2713 Cell 8 ready \u2014 Data.csv patched with all enriched layers")

In [ ]:
# ============================================================
# CELL 9 — Verify complete instance
# All required files must exist and be non-empty.
# Grid alignment check across fuels and elevation.
# ============================================================

required = {
    "fuels.asc":              "Fuel model grid (S&B codes)",
    "elevation.asc":          "Terrain elevation (m)",
    "Weather.csv":            "Hourly wind rows for C2F-W",
    "Ignitions.csv":          "Ignition cell number",
    "spain_lookup_table.csv": "S&B fuel lookup table",
    "Data.csv":               "Per-cell attributes (patched)",
}

print("── Instance file check ──────────────────────────────")
all_ok = True
for fname, desc in required.items():
    path   = INSTANCE / fname
    exists = path.exists()
    size   = path.stat().st_size / 1024 if exists else 0
    ok     = exists and size > 0
    if not ok: all_ok = False
    print(f"  {'\u2713' if ok else '\u2717'}  {fname:<30} {size:>7.1f} KB  {desc}")

print("\n── Grid alignment (fuels vs elevation) ──────────────")
_, fh = read_asc(INSTANCE / "fuels.asc")
_, eh = read_asc(INSTANCE / "elevation.asc")
for key in ["ncols", "nrows", "cellsize", "xllcorner", "yllcorner"]:
    match = abs(fh[key] - eh[key]) < 1e-3
    print(f"  {'\u2713' if match else '\u2717'}  {key:<12} "
          f"fuels={fh[key]:.2f}  elev={eh[key]:.2f}")

print(f"\n{'\u2713 Instance complete \u2014 ready for notebook 03' if all_ok else '\u2717 Fix issues above before simulating'}")